In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

#한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료!")

In [ ]:
# 각 폴더 파일 목록 확인
for dataset in ['klec', 'kspon', 'eng_eval']:
    path = f'../data/{dataset}'
    print(f"\n=== {dataset} ===")
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < 2:
            subindent = ' ' * 2 * (level + 1)
            for file in files[:5]:  # 파일 5개만 출력
                print(f'{subindent}{file}')

In [ ]:
import glob

# klec JSON 파일 하나 찾아서 열기
json_files = glob.glob('../data/klec/**/*.json', recursive=True)
print(f"총 JSON 파일 수: {len(json_files)}")
print(f"\n첫 번째 파일 경로: {json_files[0]}")

# 내용 확인
with open(json_files[0], 'r', encoding='utf-8') as f:
    sample = json.load(f)

print(f"\n=== JSON 구조 ===")
print(json.dumps(sample, ensure_ascii=False, indent=2)[:1000])  # 앞 1000자만

In [ ]:
# kspon .trn 파일 샘플 확인
with open('../data/kspon/전시문_통합_스크립트/KsponSpeech_scripts/dev.trn', 
          'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f"총 라인 수: {len(lines)}")
print("\n=== 샘플 5개 ===")
for line in lines[:5]:
    print(line.strip())

In [ ]:
# eng_eval JSON 파일 하나 열어보기
eng_files = glob.glob('../data/eng_eval/**/*.json', recursive=True)
print(f"총 JSON 파일 수: {len(eng_files)}")

with open(eng_files[0], 'r', encoding='utf-8') as f:
    eng_sample = json.load(f)

print("\n=== eng_eval JSON 구조 ===")
print(json.dumps(eng_sample, ensure_ascii=False, indent=2)[:1500])

In [ ]:
# eng_eval JSON 키 전체 확인
print("=== 최상위 키 ===")
print(list(eng_sample.keys()))

print("\n=== dialogs 첫 번째 항목 키 ===")
if 'dialogs' in eng_sample:
    print(list(eng_sample['dialogs'][0].keys()))
elif 'utterances' in eng_sample:
    print(list(eng_sample['utterances'][0].keys()))

# 점수 관련 키 찾기
print("\n=== 전체 JSON (끝부분) ===")
full_text = json.dumps(eng_sample, ensure_ascii=False, indent=2)
print(full_text[-1500:])

In [ ]:
# eng_eval 전체 점수 분포 확인
scores = []
for f in eng_files:
    with open(f, 'r', encoding='utf-8') as file:
        data = json.load(file)
        if 'rater_final' in str(data):
            # rater_final 찾기
            def find_rater(d):
                if isinstance(d, dict):
                    if 'rater_final' in d:
                        return d['rater_final']
                    for v in d.values():
                        result = find_rater(v)
                        if result: return result
                return None
            score = find_rater(data)
            if score:
                scores.append(float(score))

print(f"점수 데이터 수: {len(scores)}")
print(f"평균: {np.mean(scores):.2f}")
print(f"최소: {min(scores)}, 최대: {max(scores)}")

plt.figure(figsize=(8,4))
plt.hist(scores, bins=20, color='steelblue', edgecolor='black')
plt.title('영어 말하기 평가 점수 분포')
plt.xlabel('점수')
plt.ylabel('빈도')
plt.savefig('../results/eng_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("그래프 저장 완료!")

In [ ]:
# kspon 필러워드 비율 계산
filler_tags = ['o/', 'b/', 'n/', 'l/', 'u/']
filler_ratios = []

with open('../data/kspon/전시문_통합_스크립트/KsponSpeech_scripts/dev.trn', 
          'r', encoding='utf-8') as f:
    lines = f.readlines()

for line in lines:
    if '::' in line:
        text = line.split('::')[1].strip()
        words = text.split()
        total = len(words)
        fillers = sum(1 for w in words if any(w.endswith(tag) for tag in filler_tags))
        if total > 0:
            filler_ratios.append(fillers / total)

print(f"총 발화 수: {len(filler_ratios)}")
print(f"평균 필러워드 비율: {np.mean(filler_ratios):.3f} ({np.mean(filler_ratios)*100:.1f}%)")
print(f"중앙값: {np.median(filler_ratios):.3f}")
print(f"75% 분위: {np.percentile(filler_ratios, 75):.3f}")

plt.figure(figsize=(8,4))
plt.hist(filler_ratios, bins=30, color='coral', edgecolor='black')
plt.title('한국어 발화 필러워드 비율 분포')
plt.xlabel('필러워드 비율')
plt.ylabel('빈도')
plt.savefig('../results/filler_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("그래프 저장 완료!")